# Program-Aided Language (PAL) Prompting

**Course:** Natural Language Processing · Unit 4 — Prompt Engineering  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 12 — Prompting and In-Context Learning  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

1. Understand PAL (Gao et al., 2023): let the LLM write code instead of computing the answer
2. Send chat-completion requests to OpenRouter's free inference API
3. Compare PAL outputs against standard arithmetic prompting
4. Recognise when offloading computation to a code interpreter improves reliability


---

## 1. Environment Setup

Install once in your terminal:

```bash
pip install requests
```


In [ ]:
# pip install requests  # run once in terminal
import os

import requests


---

## 2. API Configuration

This notebook uses the [OpenRouter](https://openrouter.ai/) free tier API.  
Obtain a free key at **openrouter.ai** and store it as an environment variable:

```bash
export OPENROUTER_API_KEY=sk-or-...
```


In [ ]:
API_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "mai-ds-r1:free"

# Read key from environment — never hard-code secrets in notebooks
API_KEY = os.environ.get("OPENROUTER_API_KEY", "your-openrouter-api-key-here")


---

## 3. Helper Function


In [ ]:
def call_model(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """Send a chat-completion request to OpenRouter and return the reply text."""
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
    }
    response = requests.post(API_URL, headers=headers, json=payload)
    try:
        return response.json()["choices"][0]["message"]["content"]
    except (KeyError, IndexError):
        return f"API error: {response.text}"


---

## 4. PAL Prompt Definition

The PAL technique provides **worked examples** where each problem is solved by  
writing a Python function rather than stating the answer directly.  
The model then writes a function for the new problem, which is executed to get the answer.


In [ ]:
pal_few_shot = """
# Problem: There are 15 trees in the grove. The gardeners plant more trees today.
# After planting there are 21 trees. How many trees did they plant?
def solution():
    trees_initial = 15
    trees_after = 21
    trees_planted = trees_after - trees_initial
    return trees_planted
result = solution()
print(result)  # 6

# Problem: Julia has 23 apples. She buys 5 more and gives 7 to her friend.
# How many apples does she have now?
def solution():
    apples_start = 23
    apples_bought = 5
    apples_given = 7
    result = apples_start + apples_bought - apples_given
    return result
result = solution()
print(result)  # 21
"""

target_problem = (
    "# Problem: There are 8 students in the library. "
    "3 more arrive and 2 leave. How many students are there now?\n"
    "def solution():"
)

pal_prompt = pal_few_shot + "\n" + target_problem


---

## 5. Run the PAL Query


In [ ]:
reply = call_model(
    prompt=pal_prompt,
    system="Complete the Python function to solve the word problem.",
)
print("=== Model-generated PAL code ===")
print(reply)


> **Output interpretation:** The model should complete the `solution()` function body and (optionally) include `result = solution(); print(result)`. The expected answer is 8 + 3 − 2 = 9. PAL is more reliable than asking for a direct answer because arithmetic errors are delegated to a Python interpreter rather than the language model's attention mechanism.


---

## Summary

| Technique | How the answer is computed | Accuracy on arithmetic |
|---|---|---|
| Direct answer | Model attention | Error-prone on multi-step sums |
| Chain-of-Thought | Model step-by-step text | Better, still error-prone |
| PAL | Python function + interpreter | Near-perfect on well-specified problems |

PAL works best when problems have a clear programmatic solution. Open-ended or  
ambiguous questions still benefit from CoT or retrieval-augmented approaches.
